# Prepare HAM10000 images

Downloads the dataset, checks metadata and image integrity, and writes
consistent 224×224 RGB PNGs plus a cleaned metadata CSV. Pixel values are
normalized to `[0, 1]` when loaded with `load_processed_image`. No model
training or data splitting is performed.

In [1]:
from pathlib import Path
import kagglehub
import numpy as np
import pandas as pd
from PIL import Image, ImageOps, UnidentifiedImageError

DATASET_PATH = Path(kagglehub.dataset_download('kmader/skin-cancer-mnist-ham10000'))
OUTPUT_DIR = Path('../data/processed').resolve()
IMAGE_SIZE = (224, 224)
print(f'Dataset: {DATASET_PATH}')
print(f'Output:  {OUTPUT_DIR}')

Dataset: /Users/gaballa/.cache/kagglehub/datasets/kmader/skin-cancer-mnist-ham10000/versions/2
Output:  /Users/gaballa/Documents/projects/DiopScan/data/processed


In [2]:
# Locate the metadata and both image folders in KaggleHub's extracted dataset.
metadata_files = list(DATASET_PATH.rglob('HAM10000_metadata.csv'))
if len(metadata_files) != 1:
    raise FileNotFoundError(f'Expected one HAM10000_metadata.csv, found {len(metadata_files)}')
metadata_path = metadata_files[0]
image_roots = [p for p in DATASET_PATH.rglob('HAM10000_images*') if p.is_dir()]
image_files = {p.name: p for root in image_roots for p in root.glob('*.jpg')}
metadata = pd.read_csv(metadata_path)
required_columns = {'image_id', 'dx'}
missing_columns = required_columns - set(metadata.columns)
if missing_columns:
    raise ValueError(f'Metadata missing required columns: {sorted(missing_columns)}')
if metadata['image_id'].isna().any() or metadata['image_id'].duplicated().any():
    raise ValueError('Metadata contains missing or duplicate image_id values')
metadata['image_id'] = metadata['image_id'].astype(str)
metadata['dx'] = metadata['dx'].astype(str)
missing_images = sorted(set(metadata['image_id']) - {Path(n).stem for n in image_files})
print(f'Metadata rows: {len(metadata):,}; image files found: {len(image_files):,}')
print(f'Metadata rows without a corresponding image: {len(missing_images)}')
if missing_images:
    print('First missing IDs:', missing_images[:10])
metadata.head()

Metadata rows: 10,015; image files found: 10,015
Metadata rows without a corresponding image: 0


,lesion_id,image_id,dx,dx_type,age,sex,localization
0,HAM_0000118,ISIC_0027419,bkl,histo,80.0,male,scalp
1,HAM_0000118,ISIC_0025030,bkl,histo,80.0,male,scalp
2,HAM_0002730,ISIC_0026769,bkl,histo,80.0,male,scalp
3,HAM_0002730,ISIC_0025661,bkl,histo,80.0,male,scalp
4,HAM_0001466,ISIC_0031633,bkl,histo,75.0,male,ear


In [ ]:
# Convert to a consistent RGB format and size. Any unreadable images are
# recorded in the report and excluded from the cleaned CSV.
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
processed_dir = OUTPUT_DIR / 'images'
processed_dir.mkdir(parents=True, exist_ok=True)
records, errors = [], []
for row in metadata.itertuples(index=False):
    image_id = str(row.image_id)
    source = image_files.get(f'{image_id}.jpg')
    if source is None:
        errors.append({'image_id': image_id, 'error': 'image file not found'})
        continue
    try:
        with Image.open(source) as image:
            image = ImageOps.exif_transpose(image).convert('RGB')
            image = image.resize(IMAGE_SIZE, Image.Resampling.LANCZOS)
            image.save(processed_dir / f'{image_id}.png', format='PNG', optimize=True)
        records.append({'image_id': image_id, 'dx': str(row.dx), 'image_path': f'images/{image_id}.png'})
    except (OSError, UnidentifiedImageError, ValueError) as exc:
        errors.append({'image_id': image_id, 'error': str(exc)})

cleaned = pd.DataFrame(records)
cleaned.to_csv(OUTPUT_DIR / 'metadata.csv', index=False)
pd.DataFrame(errors, columns=['image_id', 'error']).to_csv(OUTPUT_DIR / 'preprocessing_errors.csv', index=False)
print(f'Processed: {len(cleaned):,} / {len(metadata):,}; errors/missing: {len(errors):,}')
print(f'Cleaned metadata: {OUTPUT_DIR / 'metadata.csv'}')
cleaned['dx'].value_counts()

In [ ]:
def load_processed_image(image_path):
    """Load a processed image as RGB float32 with pixel values in [0, 1]."""
    path = OUTPUT_DIR / image_path
    with Image.open(path) as image:
        array = np.asarray(image.convert('RGB'), dtype=np.float32) / 255.0
    if array.shape != (IMAGE_SIZE[1], IMAGE_SIZE[0], 3):
        raise ValueError(f'Unexpected image shape {array.shape} for {path}')
    if not np.isfinite(array).all() or array.min() < 0 or array.max() > 1:
        raise ValueError(f'Invalid normalized pixel values in {path}')
    return array

# Sanity-check one output (and report final dataset dimensions).
sample = load_processed_image(cleaned.iloc[0]['image_path']) if not cleaned.empty else None
print('Sample shape/range:', None if sample is None else (sample.shape, (float(sample.min()), float(sample.max()))))
print('Class counts:')
print(cleaned['dx'].value_counts())